In [1]:
import pandas as pd

# 读取数据 (注意路径，假设你的 notebook 在 notebook 文件夹，data 在 data 文件夹)
df = pd.read_excel("../data/datas.xlsx")

# 打印前 5 行看看
print("=== 数据前 5 行 ===")
print(df.head())

# 打印列名，确保我们没看错
print("\n=== 列名列表 ===")
print(df.columns.tolist())

=== 数据前 5 行 ===
   订单顺序编号                 订单号          用户名      商品编号     订单金额     付款金额 渠道编号  \
0       8  sys-2025-306447069  user-104863  PR000499   499.41   480.42  渠道1   
1      11  sys-2025-417411381  user-181957  PR000483   279.53   279.53  渠道1   
2      61  sys-2025-313655292  user-282453  PR000154  1658.95  1653.91  渠道1   
3      78  sys-2025-311884106  user-167776  PR000215   343.25   337.12  渠道1   
4      81  sys-2025-375273222  user-138024  PR000515   329.04   329.04  渠道1   

    平台类型                下单时间                付款时间 是否退款  
0  微信公众号 2025-01-01 01:05:50 2025-01-01 01:06:17    否  
1    APP 2025-01-01 01:36:17 2025-01-01 01:36:56    否  
2  微信公众号 2025-01-01 12:01:04 2025-01-01 12:03:20    否  
3    APP 2025-01-01 12:47:02 2025-01-01 12:47:21    否  
4    APP 2025-01-01 12:50:23 2025-01-01 12:50:50    否  

=== 列名列表 ===
['订单顺序编号', '订单号', '用户名', '商品编号', '订单金额', '付款金额', '渠道编号', '平台类型', '下单时间', '付款时间', '是否退款']


In [2]:
import pandas as pd
import numpy as np

# 1. 重新读取一份原始数据（防止刚才改乱了，从头开始）
df = pd.read_excel("../data/datas.xlsx")

print(f"🔴 清洗前数据量: {df.shape[0]} 行")

# ==========================================
# 2. 处理缺失值与重复值
# ==========================================

# 如果'订单号'是空的，这行数据就没意义，直接删
df = df.dropna(subset=['订单号'])

# 如果有重复的订单号，只保留第一条
df = df.drop_duplicates(subset=['订单号'], keep='first')

# ==========================================
# 3. 处理异常值 (针对你截图里的 bug)
# ==========================================

# 确保金额列是数字格式 (防止Excel里存成了文本)
df['订单金额'] = pd.to_numeric(df['订单金额'], errors='coerce')
df['付款金额'] = pd.to_numeric(df['付款金额'], errors='coerce')

# 【重点】删除 付款金额 > 订单金额 的异常数据
# 比如你截图里那个 订单205元，付款2044元 的明显错误
abnormal_rows = df[df['付款金额'] > df['订单金额']]
if len(abnormal_rows) > 0:
    print(f"⚠️ 发现 {len(abnormal_rows)} 条'付款金额 > 订单金额'的异常数据，已删除")
    df = df[df['付款金额'] <= df['订单金额']]

# 删除金额为负数或0的行
df = df[(df['付款金额'] > 0) & (df['订单金额'] > 0)]

# ==========================================
# 4. 时间格式转换 (非常重要！)
# ==========================================

# 将字符串时间转换为标准时间格式
df['下单时间'] = pd.to_datetime(df['下单时间'], errors='coerce')
df['付款时间'] = pd.to_datetime(df['付款时间'], errors='coerce')

# 删除时间转换失败的行
df = df.dropna(subset=['下单时间', '付款时间'])

# ==========================================
# 5. 创造新特征 (简历加分项！)
# ==========================================

# A. 计算优惠金额
df['优惠金额'] = df['订单金额'] - df['付款金额']

# B. 计算支付耗时 (秒) -> 分析用户付款快慢
df['支付耗时_秒'] = (df['付款时间'] - df['下单时间']).dt.total_seconds()

# C. 提取日期和小时 -> 用于后续按天/按小时分析
df['下单日期'] = df['下单时间'].dt.date
df['下单小时'] = df['下单时间'].dt.hour
df['星期几'] = df['下单时间'].dt.day_name() 

# ==========================================
# 6. 保存结果
# ==========================================

print(f"🟢 清洗后数据量: {df.shape[0]} 行")
print("✅ 清洗完成！正在保存...")

# 保存为 CSV (通用格式) 和 Parquet (高性能格式)
df.to_csv("../data/cleaned_orders.csv", index=False, encoding='utf-8-sig')
df.to_parquet("../data/cleaned_orders.parquet", index=False)

print("💾 文件已保存至 data/cleaned_orders.csv")
print("\n👇 最终数据预览：")
print(df.head(3))

🔴 清洗前数据量: 104557 行
⚠️ 发现 2030 条'付款金额 > 订单金额'的异常数据，已删除
🟢 清洗后数据量: 102318 行
✅ 清洗完成！正在保存...
💾 文件已保存至 data/cleaned_orders.csv

👇 最终数据预览：
   订单顺序编号                 订单号          用户名      商品编号     订单金额     付款金额 渠道编号  \
0       8  sys-2025-306447069  user-104863  PR000499   499.41   480.42  渠道1   
1      11  sys-2025-417411381  user-181957  PR000483   279.53   279.53  渠道1   
2      61  sys-2025-313655292  user-282453  PR000154  1658.95  1653.91  渠道1   

    平台类型                下单时间                付款时间 是否退款   优惠金额  支付耗时_秒  \
0  微信公众号 2025-01-01 01:05:50 2025-01-01 01:06:17    否  18.99    27.0   
1    APP 2025-01-01 01:36:17 2025-01-01 01:36:56    否   0.00    39.0   
2  微信公众号 2025-01-01 12:01:04 2025-01-01 12:03:20    否   5.04   136.0   

         下单日期  下单小时        星期几  
0  2025-01-01     1  Wednesday  
1  2025-01-01     1  Wednesday  
2  2025-01-01    12  Wednesday  
